# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [2]:
# ============================================================
# SETUP — FlyRank Warehouse
# ============================================================

!pip -q install duckdb

import duckdb
import pandas as pd
import numpy as np

# Connect DuckDB
con = duckdb.connect()

# Get your HF token from Colab Secrets
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

if not HF_TOKEN:
    raise ValueError(
        "HF_TOKEN not found. Add it in Colab: "
        "Secrets -> Add new secret -> HF_TOKEN"
    )

# Authenticate DuckDB with Hugging Face
con.execute(
    f"CREATE OR REPLACE SECRET hf_secret "
    f"(TYPE huggingface, TOKEN '{HF_TOKEN}')"
)

HF_BASE = "hf://datasets/FlyRank/internship-warehouse"

print("Hugging Face connection ready.")

Hugging Face connection ready.


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

## 1. Unit of analysis + time window

The main table for this contract is `fact_content_daily_performance`.

One row represents one pseudonymized content item for one pseudonymized client on one report date.

Therefore, the working grain is:

**one row = one client × one content item × one report date.**

The warehouse daily performance history runs from **2025-01-27 through 2026-06-30**.

For the Refresh / Content Opportunity Scoring lane, the decision is to identify content that deserves review based on observed search/content-performance signals.

The data is observational and supports directional, decision-support analysis rather than causal conclusions.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# ============================================================
# 1. VERIFY GRAIN + TIME WINDOW
# ============================================================

query = f"""
SELECT
    COUNT(*) AS total_rows,
    MIN(report_date) AS min_report_date,
    MAX(report_date) AS max_report_date,
    COUNT(DISTINCT client_hash_id) AS clients,
    COUNT(DISTINCT content_hash_id) AS content_items
FROM read_parquet(
    '{HF_BASE}/fact_content_daily_performance/**/*.parquet'
)
"""

contract_summary = con.sql(query).df()

display(contract_summary)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,min_report_date,max_report_date,clients,content_items
0,78835655,2025-01-27,2026-06-30,70,427292


### Unit of analysis

The intended analysis unit is a client-content-date observation.

I verified the warehouse rather than assuming uniqueness from the field names. The grain check found duplicate client-content-date combinations, so `client_hash_id + content_hash_id + report_date` should not be treated as a guaranteed unique primary key without investigating the additional dimensions represented by the duplicate rows.

The analysis therefore treats the warehouse as an observational fact table and uses explicit aggregation/grouping when a unique client-content-date view is required.

In [4]:
# Check whether client + content + date is unique

query = f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT (
        report_date,
        client_hash_id,
        content_hash_id
    )) AS distinct_grain_keys
FROM read_parquet(
    '{HF_BASE}/fact_content_daily_performance/**/*.parquet'
)
"""

grain_check = con.sql(query).df()

display(grain_check)

total_rows = grain_check.loc[0, "total_rows"]
distinct_keys = grain_check.loc[0, "distinct_grain_keys"]

print("Total rows:", total_rows)
print("Distinct grain keys:", distinct_keys)
print("Duplicate observations:", total_rows - distinct_keys)

if total_rows == distinct_keys:
    print("PASS: grain is unique.")
else:
    print(
        "NOTE: client + content + report_date is not fully unique. "
        "Additional dimensions exist in the fact table."
    )


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,distinct_grain_keys
0,78835655,78829265


Total rows: 78835655
Distinct grain keys: 78829265
Duplicate observations: 6390
NOTE: client + content + report_date is not fully unique. Additional dimensions exist in the fact table.


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

## 2. Fields: feature / label / context / excluded

### Features

The following observed fields are candidates for the feature set:

- `impressions`
- `clicks`
- `avg_position`
- `sessions`
- `sessions_ai`
- `scroll_events`
- `word_count`
- `content_age_days`

These represent observable search, traffic, engagement, or content-lifecycle measurements.

### Label / proxy

For this data contract, the eventual decision target is a **future content-performance outcome**, such as future decline or recovery.

The target is not taken from a product decision field. It should be constructed from a future time window after the feature window.

### Context

- `client_hash_id` — identifies the pseudonymized client for grouping and validation.
- `content_hash_id` — identifies the pseudonymized content item.
- `report_date` — defines the observation date.
- `ga4_data_available` — indicates whether GA4 measurements are available for the row.

These fields provide context and should not automatically become predictive features.

### Excluded

- `client_hash_id` and `content_hash_id` as raw numeric/model features — they are identifiers.
- `report_date` as a raw model feature — it defines time rather than page behaviour.
- `trend_pct` / `trend_direction` when used as a target-derived decision signal — they can create circularity if the target is based on the same movement.
- Any future-window metric — unavailable at the decision moment and therefore leakage.
- Any product-generated score or action flag — these represent an existing decision rather than an independent signal.
- Raw client names, URLs, queries, titles, or other identifying information — these are not part of the public warehouse release.

The feature set will therefore contain only information that would be available at the decision moment.

In [5]:

# ============================================================
# 2. FIELD BUCKETS
# ============================================================

# Inspect the actual schema first.
schema_query = f"""
DESCRIBE
SELECT *
FROM read_parquet(
    '{HF_BASE}/fact_content_daily_performance/**/*.parquet'
)
"""

schema_df = con.sql(schema_query).df()

print("Warehouse columns:")
display(schema_df)

Warehouse columns:


,column_name,column_type,null,key,default,extra
0,report_date,DATE,YES,None,None,None
1,client_hash_id,VARCHAR,YES,None,None,None
2,content_hash_id,VARCHAR,YES,None,None,None
3,client_has_gsc,BOOLEAN,YES,None,None,None
4,client_has_ga4,BOOLEAN,YES,None,None,None
5,gsc_data_available,BOOLEAN,YES,None,None,None
6,ga4_data_available,BOOLEAN,YES,None,None,None
7,gsc_impressions,BIGINT,YES,None,None,None
8,gsc_clicks,BIGINT,YES,None,None,None
9,gsc_sum_position,BIGINT,YES,None,None,None


In [6]:
# ============================================================
# FIELD BUCKET CHECK
# ============================================================

available_fields = set(schema_df["column_name"])

feature_candidates = [
    "impressions",
    "clicks",
    "avg_position",
    "sessions",
    "sessions_ai",
    "scroll_events",
    "word_count",
    "content_age_days"
]

context_fields = [
    "client_hash_id",
    "content_hash_id",
    "report_date",
    "ga4_data_available"
]

excluded_candidates = [
    "trend_pct",
    "trend_direction",
    "health_score",
    "priority_score",
    "action_type",
    "refresh_tier"
]

print("FEATURES PRESENT:")
print([x for x in feature_candidates if x in available_fields])

print("\nCONTEXT FIELDS PRESENT:")
print([x for x in context_fields if x in available_fields])

print("\nPOTENTIAL EXCLUSIONS PRESENT:")
print([x for x in excluded_candidates if x in available_fields])

FEATURES PRESENT:
['sessions_ai', 'scroll_events']

CONTEXT FIELDS PRESENT:
['client_hash_id', 'content_hash_id', 'report_date', 'ga4_data_available']

POTENTIAL EXCLUSIONS PRESENT:
[]


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

## 3. Verify it with queries (grain, counts, missing values, windows)

The contract is verified directly against the warehouse rather than relying only on documentation.

The checks below verify:

1. the row grain;
2. total rows and distinct clients/content items;
3. the observed date window;
4. missingness for important measurements;
5. GA4 availability.

The measurements are treated as observed properties of this snapshot. Missing GA4 values are not interpreted as zero traffic because GA4 coverage is uneven across the history.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# ============================================================
# 3A. ROW COUNTS + DATE WINDOW
# ============================================================

query = f"""
SELECT
    COUNT(*) AS rows,
    COUNT(DISTINCT client_hash_id) AS clients,
    COUNT(DISTINCT content_hash_id) AS content_items,
    MIN(report_date) AS first_date,
    MAX(report_date) AS last_date
FROM read_parquet(
    '{HF_BASE}/fact_content_daily_performance/**/*.parquet'
)
"""

counts_df = con.sql(query).df()

display(counts_df)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,rows,clients,content_items,first_date,last_date
0,78835655,70,427292,2025-01-27,2026-06-30


In [8]:
# ============================================================
# 3B. MONTHLY COVERAGE
# ============================================================

query = f"""
SELECT
    DATE_TRUNC('month', report_date) AS month,
    COUNT(*) AS rows,
    COUNT(DISTINCT client_hash_id) AS clients,
    COUNT(DISTINCT content_hash_id) AS content_items
FROM read_parquet(
    '{HF_BASE}/fact_content_daily_performance/**/*.parquet'
)
GROUP BY 1
ORDER BY 1
"""

monthly_df = con.sql(query).df()

display(monthly_df)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,month,rows,clients,content_items
0,2025-01-01,1297,2,476
1,2025-02-01,75985,3,5903
2,2025-03-01,167859,4,10374
3,2025-04-01,285114,4,13046
4,2025-05-01,349923,4,14887
5,2025-06-01,329201,9,16399
6,2025-07-01,469794,16,27945
7,2025-08-01,704962,15,37204
8,2025-09-01,845813,23,53127
9,2025-10-01,2165471,31,110339


In [9]:
# ============================================================
# 3C. MISSING VALUES
# ============================================================

query = f"""
SELECT
    COUNT(*) AS total_rows,

    COUNT(gsc_impressions) AS impressions_present,
    COUNT(gsc_clicks) AS clicks_present,
    COUNT(gsc_avg_position) AS avg_position_present,
    COUNT(sessions_ai) AS sessions_ai_present,
    COUNT(scroll_events) AS scroll_events_present

FROM read_parquet(
    '{HF_BASE}/fact_content_daily_performance/**/*.parquet'
)
"""

missing_df = con.sql(query).df()

display(missing_df)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,impressions_present,clicks_present,avg_position_present,sessions_ai_present,scroll_events_present
0,78835655,78737649,78737649,28970001,49200328,49200328


In [11]:
# ============================================================
# 3D. GA4 AVAILABILITY
# ============================================================

query = f"""
SELECT
    ga4_data_available,
    COUNT(*) AS rows
FROM read_parquet(
    '{HF_BASE}/fact_content_daily_performance/**/*.parquet'
)
GROUP BY ga4_data_available
ORDER BY ga4_data_available
"""

ga4_df = con.sql(query).df()

display(ga4_df)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,ga4_data_available,rows
0,False,46383873
1,True,2816455
2,<NA>,29635327


In [13]:
# ============================================================
# 3E. GRAIN CHECK ON SAMPLE TABLE
# ============================================================

sample_path = (
    f"{HF_BASE}/fact_content_daily_performance_sample.parquet"
)

query = f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT
        CONCAT(
            CAST(report_date AS VARCHAR), '|',
            client_hash_id, '|',
            content_hash_id
        )
    ) AS distinct_grain_keys,
    MIN(report_date) AS first_date,
    MAX(report_date) AS last_date
FROM read_parquet('{sample_path}')
"""

sample_grain_df = con.sql(query).df()

display(sample_grain_df)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,distinct_grain_keys,first_date,last_date
0,11694072,11687682,2026-06-01,2026-06-30


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

## 4. Data limits

This dataset has several important limitations.

### 1. Unbalanced history

Clients do not have identical history lengths. A page or client appearing in the dataset does not imply that it has observations for every date.

Therefore, missing dates should not automatically be interpreted as zero performance.

### 2. GSC-only early rows

Some early rows have search-console measurements but no GA4 measurements because GA4 tracking started later for those clients.

Therefore, missing GA4 sessions are not equivalent to zero sessions.

### 3. Window overlaps

Feature and target windows must be separated.

For a future-outcome task, features should come from a prior window and the label should come from a later window. Reusing the same dates in both windows can leak information from the target into the features.

### 4. Sparse measurements

Clicks, sessions, AI sessions, and engagement measurements are not available for every daily row. Low-volume measurements can therefore be noisy.

### 5. Observational data

The dataset records what happened. It does not prove that changing a page caused its future performance to change.

The results should therefore be described as observed, measured, directional, and decision-support evidence rather than causal proof.

### 6. Pseudonymized data

Client, content, URL, and keyword identifiers are pseudonymized. The data should not be used to attempt re-identification, and no client-identifying information should be published.

In [14]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# ============================================================
# 4. DATA LIMIT CHECKS
# ============================================================

# ------------------------------------------------------------
# A. Check history coverage by client
# ------------------------------------------------------------

query = f"""
SELECT
    client_hash_id,
    MIN(report_date) AS first_date,
    MAX(report_date) AS last_date,
    COUNT(DISTINCT report_date) AS observed_days
FROM read_parquet(
    '{HF_BASE}/fact_content_daily_performance/**/*.parquet'
)
GROUP BY client_hash_id
ORDER BY first_date
"""

client_history_df = con.sql(query).df()

display(client_history_df.head(20))

print(
    "Clients with different history lengths:",
    client_history_df["observed_days"].nunique()
)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,client_hash_id,first_date,last_date,observed_days
0,client_ff644d8251367cbb,2025-01-27,2026-06-30,514
1,client_9958f0a7ae1df715,2025-01-27,2026-06-30,520
2,client_73cda7b4e4f265ea,2025-02-11,2026-06-30,505
3,client_fef1a8f436438636,2025-03-11,2026-06-30,477
4,client_62f4a7e64f5e0096,2025-06-07,2026-06-30,389
5,client_b10cb2997d0c7c86,2025-06-18,2026-06-30,378
6,client_65de48885f4ef01b,2025-06-21,2026-06-30,375
7,client_c182d11e4862a37d,2025-06-21,2026-06-30,375
8,client_3197e6291363b4db,2025-06-29,2026-06-30,367
9,client_625b6439094e23e4,2025-07-01,2026-06-30,252


Clients with different history lengths: 47


In [15]:
# ------------------------------------------------------------
# B. Check GSC-only / GA4-unavailable rows
# ------------------------------------------------------------

query = f"""
SELECT
    ga4_data_available,
    COUNT(*) AS rows,
    MIN(report_date) AS first_date,
    MAX(report_date) AS last_date
FROM read_parquet(
    '{HF_BASE}/fact_content_daily_performance/**/*.parquet'
)
GROUP BY ga4_data_available
ORDER BY ga4_data_available
"""

ga4_history_df = con.sql(query).df()

display(ga4_history_df)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,ga4_data_available,rows,first_date,last_date
0,False,46383873,2025-01-27,2026-06-30
1,True,2816455,2025-10-29,2026-06-30
2,<NA>,29635327,2025-11-05,2026-06-30


In [16]:
# ------------------------------------------------------------
# C. Check date-window overlap example
# ------------------------------------------------------------

# Example contract:
# Feature window = February 2026
# Target window  = March 2026

feature_start = "2026-02-01"
feature_end   = "2026-02-28"

target_start = "2026-03-01"
target_end   = "2026-03-31"

assert feature_end < target_start

print("Feature window:", feature_start, "to", feature_end)
print("Target window :", target_start, "to", target_end)
print("PASS: feature and target windows do not overlap.")

Feature window: 2026-02-01 to 2026-02-28
Target window : 2026-03-01 to 2026-03-31
PASS: feature and target windows do not overlap.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.